In [8]:
import glob
import os
import re
import pandas as pd
import numpy as np
from PIL import Image
import random

random.seed(0)

# --- 1) 色 → クラス名 ---
color2name = {
    (0, 50, 128): "hepatic_vein",
    (111, 74, 0): "liver_ligament",
    (127, 127, 127): "black_background",
    (169, 255, 184): "l_hook_electrocautery",
    (170, 255, 0): "grasper",
    (186, 183, 75): "fat",
    (210, 140, 140): "abdominal_wall",
    (231, 70, 156): "gi_tract",
    (255, 0, 0): "blood",
    (255, 85, 0): "connective_tissue",
    (255, 114, 114): "liver",
    (255, 160, 165): "gallbladder",
    (255, 255, 0): "cystic_duct_yellow",
    (255, 255, 255): "cystic_duct_white",
}

# --- 2) phase & tool annotation のパス ---
phase_dir = "../data/cholec80/phase_annotations"
tool_dir  = "../data/cholec80/tool_annotations"

# --- 3) マスク画像 ---
mask_paths = glob.glob("../data/cholec80/seg/*/*/frame_*_endo_color_mask.png")
mask_paths = random.sample(mask_paths, 30)

def parse_video_frame(path):
    """
    例: ../data/cholec80/seg/train/video05/frame_00100_endo_color_mask.png
    → video=5, frame=100
    """
    v = re.search(r"video(\d+)", path)
    f = re.search(r"frame_(\d+)_endo", path)
    return int(v.group(1)), int(f.group(1))

# --- 4) 全体行を入れる ---
rows = []

for path in mask_paths:
    video, frame = parse_video_frame(path)
    
    # --- マスク画像の集計 ---
    img = Image.open(path)
    arr = np.array(img)
    unique_colors, counts = np.unique(arr.reshape(-1, 3), axis=0, return_counts=True)

    row = {name: 0 for name in color2name.values()}
    row["mask_path"] = path
    row["video"] = video
    row["frame"] = frame
    
    for col, cnt in zip(unique_colors, counts):
        col_t = tuple(col.tolist())
        if col_t in color2name:
            row[color2name[col_t]] = int(cnt)

    # --- 5) フェーズアノテーション ---
    phase_file = f"{phase_dir}/video{video:02d}-phase.txt"
    df_phase = pd.read_csv(phase_file, sep="\t")
    
    # frame に一致する行を抽出
    phase = df_phase.loc[df_phase["Frame"] == frame, "Phase"]
    if len(phase) > 0:
        row["phase"] = phase.values[0]
    else:
        row["phase"] = "Unknown"

    # --- 6) ツールアノテーション ---
    tool_file = f"{tool_dir}/video{video:02d}-tool.txt"
    df_tool = pd.read_csv(tool_file, sep="\t")
    
    tool_row = df_tool.loc[df_tool["Frame"] == frame]
    if len(tool_row) > 0:
        for col in ["Grasper","Bipolar","Hook","Scissors","Clipper","Irrigator","SpecimenBag"]:
            row[col] = int(tool_row[col].values[0])
    else:
        for col in ["Grasper","Bipolar","Hook","Scissors","Clipper","Irrigator","SpecimenBag"]:
            row[col] = 0

    rows.append(row)


In [9]:
# --- 7) DataFrame → CSV ---
df = pd.DataFrame(rows)
df.to_csv("../data/cholec80_sampled_dataset.csv", index=False)
df

,hepatic_vein,liver_ligament,black_background,l_hook_electrocautery,grasper,fat,abdominal_wall,gi_tract,blood,connective_tissue,...,video,frame,phase,Grasper,Bipolar,Hook,Scissors,Clipper,Irrigator,SpecimenBag
0,0,0,123552,0,744,32577,108510,12561,0,0,...,26,1975,Preparation,1,0,0,0,0,0,0
1,104,0,143355,18966,1716,59449,27226,0,0,47957,...,12,19572,GallbladderDissection,0,0,0,0,0,0,0
2,0,0,71688,0,7455,40314,782,2826,9025,47116,...,1,16493,CalotTriangleDissection,0,0,0,0,0,0,0
3,0,0,164329,0,0,0,213420,215,0,0,...,24,594,Preparation,0,0,0,0,0,0,0
4,0,0,0,10682,0,65338,30659,9144,0,47946,...,12,19669,GallbladderDissection,0,0,0,0,0,0,0
5,0,0,96254,0,7105,83979,170462,3774,0,0,...,27,452,Preparation,0,0,0,0,0,0,0
6,0,0,94117,0,16090,77469,136472,5214,0,0,...,17,1836,Preparation,0,0,0,0,0,0,0
7,0,0,164401,0,6973,74273,12526,23995,0,0,...,24,9985,Preparation,0,0,0,0,0,0,0
8,0,0,113465,0,10716,87230,155197,0,0,0,...,43,505,Preparation,0,0,0,0,0,0,0
9,0,0,113621,0,0,69882,176873,0,0,0,...,43,67,Preparation,0,0,0,0,0,0,0
